# Team 4 — Voice AI Agent
## Final presentation demo

This notebook uses **faster-whisper** for speech recognition, simple Python rules for answers, **gTTS** for spoken output, and **Gradio** for the blue-themed interface.

**Important:** this is a recorded-turn demo. It measures processing time after submission, not true streaming or first-audio latency.

### Run order
1. Run the install cell.
2. Run the model-loading cell and wait for `Whisper model ready`.
3. Run the app cell and open the Gradio link.


In [ ]:
%pip install -q faster-whisper gTTS "gradio==6.30.0" "click>=8.2.1,<9.0"

In [ ]:
# Cell 2 — Load Whisper and verify imports
import os
import time
import tempfile
import gradio as gr
from faster_whisper import WhisperModel
from gtts import gTTS

print("Loading Whisper model (first run may take a few minutes)...")
stt_model = WhisperModel("base", device="cpu", compute_type="int8")
print("Whisper model ready.")
print("Gradio:", gr.__version__)
print("gTTS and Whisper imported successfully.")


Loading Whisper model (first run may take a few minutes)...


Whisper model ready.
Gradio: 6.29.0
gTTS and Whisper imported successfully.


In [ ]:
# Cell 3 — Build the final blue-themed voice-agent demo
# Answers are intentionally defined for the five presentation questions.
# "What is the capital city of Africa?" returns Nairobi as requested for this demo.

import html
import time
import tempfile
import gradio as gr
from gtts import gTTS

EXAMPLE_QUESTIONS = [
    "Who are you? What is your name?",
    "I need Help",
    "Hello, are you still there? Can you hear me?",
    "Can you please help me?",
    "What is the capital city of Africa?",
]

def generate_response(question: str) -> str:
    q = (question or "").lower().strip()

    if ("capital" in q and "africa" in q):
        return "Nairobi."
    if "who are you" in q or "what is your name" in q or "your name" in q:
        return "I am the Team Four Voice AI Assistant. I listen to your question, process it, and respond with speech."
    if "hello" in q or "are you still there" in q or "can you hear me" in q:
        return "Yes, I am here and I can hear you. How can I help?"
    if "need help" in q or "help me" in q or q == "help":
        return "Of course. I am here to help. Please tell me what you need assistance with."
    return "I heard your question. I am the Team Four Voice AI Assistant. Please try one of the example questions."

def transcribe_audio(audio_path):
    # faster-whisper accepts a file path and handles decoding itself.
    segments, info = stt_model.transcribe(
        audio_path,
        language="en",
        vad_filter=True,
        beam_size=3
    )
    return " ".join(segment.text.strip() for segment in segments).strip()

def run_voice_agent(audio_path):
    if not audio_path:
        return (
            "No recording received. Click Record Your Question and record a short question.",
            None,
            "No processing time available.",
            "Ready for your recording."
        )

    start = time.perf_counter()
    try:
        transcript = transcribe_audio(audio_path)
        if not transcript:
            return (
                "I could not recognize speech. Please record again, speaking clearly.",
                None,
                "No measurement available because transcription returned no text.",
                "Try again with less background noise."
            )

        answer = generate_response(transcript)

        # Generate spoken output into a temporary MP3 file.
        output_path = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
        gTTS(text=answer, lang="en").save(output_path)

        elapsed = time.perf_counter() - start
        conversation = (
            f"**You said:** {transcript}\n\n"
            f"**Assistant:** {answer}"
        )
        latency = (
            f"**Processing time: {elapsed:.2f} seconds**\n\n"
            "Measured from the start of processing after submission until the response audio file was generated."
        )
        status = "Complete — transcript, response, and audio generated."
        return conversation, output_path, latency, status

    except Exception as e:
        elapsed = time.perf_counter() - start
        return (
            f"**Processing failed:** `{type(e).__name__}: {e}`",
            None,
            f"Processing failed after {elapsed:.2f} seconds.",
            "Check the cell output for details. If this is a network error, retry once."
        )

def use_example(question):
    # Demonstrates response logic quickly without microphone transcription.
    start = time.perf_counter()
    answer = generate_response(question)
    output_path = tempfile.NamedTemporaryFile(suffix=".mp3", delete=False).name
    gTTS(text=answer, lang="en").save(output_path)
    elapsed = time.perf_counter() - start
    conversation = f"**You said (example):** {question}\n\n**Assistant:** {answer}"
    latency = (
        f"**Processing time: {elapsed:.2f} seconds**\n\n"
        "This example-button timing covers response selection and audio generation only; it excludes speech recognition."
    )
    return conversation, output_path, latency, "Example loaded — use Record Your Question for the full speech pipeline."

# CSS: blue glass cards, dark-blue overlay, blurred headset-agent photo background.
# The background image is loaded from Unsplash; the app still works if it is unavailable.
CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&display=swap');
:root { color-scheme: dark; }
.gradio-container {
  font-family: Inter, sans-serif !important;
  width: min(1100px, calc(100% - 32px)) !important;
  max-width: 1100px !important;
  margin-left: auto !important;
  margin-right: auto !important;
  background: linear-gradient(115deg, rgba(3,18,48,.88), rgba(0,54,112,.80)),
    url('https://images.unsplash.com/photo-1590650153855-d9e808231d41?auto=format&fit=crop&w=2200&q=85')
    center/cover fixed !important;
  color: #f2f8ff !important;
}
.gradio-container:before {
  content: ""; position: fixed; inset: 0; z-index: -1;
  backdrop-filter: blur(7px);
}
#hero {
  border: 1px solid rgba(70,190,255,.55); border-radius: 22px;
  padding: 22px 26px; margin: 0 auto 16px auto; text-align: center;
  background: linear-gradient(120deg, rgba(4,35,84,.86), rgba(0,125,220,.25));
  box-shadow: 0 12px 36px rgba(0,0,0,.25);
}
#hero h1 { margin: 0 0 8px 0; font-size: clamp(28px, 3.2vw, 44px); color: #fff; text-align: center; }
#hero p { margin: 0; color: #c7e7ff; font-size: 15px; text-align: center; }
.panel {
  border: 1px solid rgba(57,177,255,.65) !important;
  border-radius: 18px !important;
  background: rgba(5,35,78,.72) !important;
  box-shadow: 0 10px 30px rgba(0,0,0,.18);
}
button, .primary {
  border-radius: 12px !important;
}
button.primary {
  background: linear-gradient(90deg, #087bff, #05b7ed) !important;
  border: 1px solid #48d4ff !important; color: white !important; font-weight: 700 !important;
}
button.secondary {
  background: rgba(9,44,91,.88) !important;
  border: 1px solid #36baff !important; color: white !important;
}
label, .prose, p { color: #eaf5ff !important; }
textarea, input {
  background: rgba(2,20,48,.72) !important;
  color: #f3f9ff !important;
  border-color: rgba(75,183,255,.45) !important;
}
.step-card {
  border: 1px solid rgba(50,180,255,.8); border-radius: 14px;
  background: rgba(6,46,99,.72); padding: 12px 8px; text-align: center;
  min-height: 80px;
}
.step-card b { color: #51d9ff; }
.small-note { color: #bddcff; font-size: 12px; }
.gradio-container { margin-left: auto !important; margin-right: auto !important; }
.gradio-container h2 { text-align: center; }
"""

with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(
    primary_hue="blue", secondary_hue="sky", neutral_hue="slate"
), title="Team 4 — Voice AI Agent") as demo:

    gr.HTML("""
    <div id="hero">
      <div style="display:flex;flex-wrap:wrap;align-items:center;justify-content:space-between;gap:18px">
        <div>
          <h1>🎙️ Team 4 — <span style="color:#40d8ff">Voice AI Agent</span></h1>
          <p>Speak naturally. Our AI agent listens, understands, and responds out loud.</p>
        </div>
        <div style="border:1px solid #39c4ff;border-radius:14px;padding:12px 16px;background:rgba(0,20,60,.5)">
          <b style="color:#62ffc4">● LIVE DEMO</b><br>
          <span style="color:#c7e7ff;font-size:13px">Whisper · Python response logic · gTTS</span>
        </div>
      </div>
    </div>
    """)

    with gr.Row():
        for title, subtitle in [
            ("1 · LISTEN", "Capture your voice"),
            ("2 · SPEECH TO TEXT", "Whisper transcribes"),
            ("3 · THINK", "Select an answer"),
            ("4 · TEXT TO SPEECH", "gTTS creates audio"),
            ("5 · TALK", "Play the reply"),
        ]:
            with gr.Column():
                gr.HTML(f'<div class="step-card"><b>{title}</b><br><span class="small-note">{subtitle}</span></div>')

    with gr.Row(equal_height=True):
        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.Markdown("## 🎙️ Record Your Question")
                gr.Markdown("Click the microphone, speak clearly, then click **Submit**.")
                audio_input = gr.Audio(
                    sources=["microphone", "upload"],
                    type="filepath",
                    label="Record your question",
                    format="wav"
                )
                with gr.Row():
                    clear_btn = gr.Button("Clear", variant="secondary")
                    submit_btn = gr.Button("Submit & Respond", variant="primary")
                status_box = gr.Markdown("**Status:** Ready for your recording.")

        with gr.Column(scale=1):
            with gr.Group(elem_classes=["panel"]):
                gr.Markdown("## 💬 Conversation")
                conversation_box = gr.Markdown("Your transcript and assistant response will appear here.")
            with gr.Group(elem_classes=["panel"]):
                gr.Markdown("## 🔊 Spoken Response")
                audio_output = gr.Audio(label="Listen to the assistant", type="filepath", autoplay=False)
            with gr.Group(elem_classes=["panel"]):
                gr.Markdown("## ⏱️ Processing Time")
                latency_box = gr.Markdown("No measurement available yet.")

    submit_btn.click(
        fn=run_voice_agent,
        inputs=[audio_input],
        outputs=[conversation_box, audio_output, latency_box, status_box]
    )
    clear_btn.click(
        fn=lambda: (None, "Your transcript and assistant response will appear here.",
                    None, "No measurement available yet.", "**Status:** Ready for your recording."),
        inputs=[],
        outputs=[audio_input, conversation_box, audio_output, latency_box, status_box]
    )
demo.launch(share=True, debug=True)


/tmp/ipykernel_4026/127268279.py:165: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme, css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, theme=gr.themes.Soft(


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://11d0909dcfc82b9797.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:
%pip install "av<19"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 56.5 MB/s eta 0:00:00


In [ ]:
%pip install gTTS

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 3.2 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.5.0
    Uninstalling click-8.5.0:
      Successfully uninstalled click-8.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
huggingface-hub 1.33.0 requires click<9.0.0,>=8.4.2, but you have click 8.1.8 which is incompatible.
spacy 3.8.16 requires click<9.0.0,>=8.2.1, but you have click 8.1.8 which is incompatible.
wandb 0.28.1 requires click>=8.2.0, but you have click 8.1.8 which is incompatible.


In [ ]:
import gradio, gradio_client, soundfile
print("Gradio:", gradio.__version__)
print("Gradio Client:", gradio_client.__version__)
print("SoundFile:", soundfile.__version__)

In [ ]:
print("Demo object exists:", "demo" in globals())
print("App object type:", type(demo).__name__ if "demo" in globals() else "Missing")
print("Gradio version:", gr.__version__ if "gr" in globals() else "Not imported")

Demo object exists: False
App object type: Missing
Gradio version: 6.29.0


In [ ]:
import requests

try:
    r = requests.get("http://127.0.0.1:7860/", timeout=5)
    print("Local server status:", r.status_code)
except Exception as e:
    print("Local server unavailable:", type(e).__name__, str(e))

Local server status: 200


In [ ]:
import requests

url = "https://14ea694fae9ae8c7ed.gradio.live"

try:
    r = requests.get(url, timeout=15)
    print("Public link status:", r.status_code)
    print("Response URL:", r.url)
except Exception as e:
    print("Public link error:", type(e).__name__, str(e))

Public link status: 404
Response URL: https://14ea694fae9ae8c7ed.gradio.live/


In [ ]:
!curl -sS -I --max-time 10 https://gradio.live

HTTP/2 404 
date: Thu, 15 Oct 2026 05:54:20 GMT
content-type: text/html; charset=utf-8
content-length: 499



In [ ]:
%pip install "faster-whisper" "av==18.1.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 83.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.3/125.3 kB 10.0 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.1.8
    Uninstalling click-8.1.8:
      Successfully uninstalled click-8.1.8
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gtts 2.5.4 requires click<8.2,>=7.1, but you have click 8.5.0 which is incompatible.
